# End of week Exercise - Generating Synthentic Data - week 3

The intent is to build an interactive synthentic dataset generator. 
This work will use a variety of different models for implementation.

Problem statement: To build an interactive synthentic dataset generator that responsd with sample data and download link for the entire dataset.

In [76]:
%pip install tabulate


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [21]:
%pip install gradio pandas faker openai python-dotenv

/Users/olugbengafalodun/projects/llm_engineering/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr # oh yeah!
from IPython.display import Markdown, display
import uuid
import pandas as pd
from faker import Faker

In [2]:
# constants
models =  [ "openai/gpt-4o-mini", "openai/gpt-4.1-mini", "openai/gpt-4o", "openai/gpt-5-nano", "anthropic/claude-3-5-sonnet", 
            "anthropic/claude-3-opus", "google/gemini-1.5-pro", "meta-llama/llama-3.1-70b-instruct", "mistralai/mistral-large",  
            "deepseek/deepseek-chat"]

SYSTEM_PROMPT = """
You are a Senior Data Engineer and Synthetic Data Generation Specialist.

Your role is to help users design and generate high-quality synthetic datasets based on their prompts.

You have deep expertise in:
1. Data modeling
2. Dataset schema design
3. Synthetic data generation
4. Statistical realism
5. Business domain datasets (finance, healthcare, retail, logistics, SaaS, telecom, etc.)
6. Data formats such as CSV, JSON, SQL tables, and Parquet.

You have access to tools that can generate datasets. Use these tools whenever appropriate.

CORE RESPONSIBILITIES

1. Understand the user's intent
    - Carefully interpret the user's request.
    - Identify the business domain, dataset type, and expected use case (e.g., ML training, analytics, testing, demo data).

2. Ask clarifying questions when necessary
    If the request lacks important details, ask questions such as:
    - number of rows
    - columns or schema
    - data format (CSV, JSON, SQL, etc.)
    - statistical distribution
    - constraints or relationships between fields
    - industry or business context

3. Infer intent when possible
    If the user’s request is partially ambiguous:
    - infer the most likely interpretation
    - ask the user to confirm before generating the dataset if the assumption could significantly affect the result
    - explain your assumption

4. Design realistic datasets
    Generated datasets should:
    - follow realistic business logic
    - maintain consistent relationships between fields
    - avoid impossible combinations
    - use plausible values, distributions, and timestamps

5. Be structured and explicit
    When generating a dataset:
    - clearly define the schema
    - explain the fields and data types
    - generate sample rows
    - ensure formatting is valid and consistent

6. Adapt to the user’s technical level
    - Provide concise responses for experienced users
    - Provide more explanation for beginners

7. Use Tools for Dataset Generation
    When generating datasets:
    - use available tools to generate the full dataset
    - avoid manually fabricating large datasets in chat
    - generate only a sample preview in the response

RESPONSE STRUCTURE

When providing a dataset, structure the response using the following format:

1. Dataset Description

Brief explanation of what the dataset represents.

2. Schema

List all fields with:
- Column Name
- Data Type
- Description

3. Sample Data

Provide a small preview sample (5–10 rows) of the generated dataset.

4. Download Link

Provide a download link for the complete dataset generated by the tool.

The dataset download may be provided in formats such as:
- CSV
- JSON
- Parquet
- Excel

VALID JSON RESPONSE FORMAT (CRITICAL)

When generating the dataset schema for the system to process, you MUST return a valid JSON object using the following format:

{
  "rows": <number_of_rows>,
  "schema": {
    "column_name_1": "datatype",
    "column_name_2": "datatype",
    "column_name_3": "datatype"
  }
}

Rules:
- Always include the key "rows"
- Always include the key "schema"
- The schema must contain at least 3 columns
- Column names must be snake_case
- Values must be valid datatype labels

Allowed datatypes include:
name
email
phone
city
date
amount
uuid
text
integer
float
boolean

Do NOT return explanations, markdown, or text outside the JSON object when generating the schema.

INTERACTION GUIDELINES
- Be clear, structured, and concise.
- Prefer clarity over verbosity.
- Always prioritize data realism and usability.
- Ask clarification questions if the request is underspecified.
- Generate only sample rows in chat, and provide a download link for the complete dataset.
"""

In [3]:
# Load environment variables in a file called .env
# Print the key prefixes to help with any debugging
# You can choose whichever providers you like - or all Ollama

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
openrouter_base_url = os.getenv('OPENROUTER_BASE_URL')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:3]}")
else:
    print("OpenRouter API Key not set (and this is optional)")

OpenAI API Key exists and begins sk-proj-
OpenRouter API Key exists and begins sk-


In [4]:
# Connect to OpenAI, Anthropic and Google; comment out the Claude or Google lines if you're not using them

# openai = OpenAI()

openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_base_url)

In [5]:
def build_payload(prompt: str, model: str) -> str:
    payload = {}
    if model in models:
        payload = {
            "model": model,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt}
            ],
            "temperature": 0.3,
            "response_format": {"type": "json_object"}
        }
    else:
        raise ValueError(f"Invalid model: {model}")

    return payload

In [6]:
def build_stream_payload(prompt: str, model: str) -> str:
    payload = {}
    if model in models:
        payload = {
            "model": model,
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt}
            ],
            "temperature": 0.3,
            "stream": True,
            "response_format": {"type": "json_object"}
        }
    else:
        raise ValueError(f"Invalid model: {model}")

    return payload

In [7]:
fake = Faker()

def generate_dataset(schema, rows=100):

    data = []

    for _ in range(rows):

        row = {}

        for column, dtype in schema.items():

            if dtype == "name":
                row[column] = fake.name()

            elif dtype == "email":
                row[column] = fake.email()

            elif dtype == "phone":
                row[column] = fake.phone_number()

            elif dtype == "city":
                row[column] = fake.city()

            elif dtype == "date":
                row[column] = fake.date()

            elif dtype == "amount":
                row[column] = round(fake.random_number(digits=4), 2)

            elif dtype == "uuid":
                row[column] = str(uuid.uuid4())

            else:
                row[column] = fake.word()

        data.append(row)

    df = pd.DataFrame(data)

    # Create writable directory
    output_dir = "datasets"
    os.makedirs(output_dir, exist_ok=True)

    filename = os.path.join(output_dir, f"dataset_{uuid.uuid4().hex}.csv")

    df.to_csv(filename, index=False)

    return df, filename

In [8]:
def invoke_model(question, model):

    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question}
        ],
        "response_format": {"type": "json_object"}
    }

    try:
        response = openrouter.chat.completions.create(**payload)

        message = response.choices[0].message.content

        # Safely parse JSON
        try:
            schema_info = json.loads(message)
        except json.JSONDecodeError:
            return "Model returned invalid JSON."

        # Validate schema
        if "schema" not in schema_info:
            return f"Invalid schema response:\n\n{schema_info}"

        rows = schema_info.get("rows", 100)
        schema = schema_info["schema"]

        # Generate dataset
        df, file_path = generate_dataset(schema, rows)

        preview = df.head(10).to_markdown()

        response_text = f"""
## Dataset Generated Successfully

**Total Rows:** {rows}

### Schema

```json
{json.dumps(schema, indent=2)}
```

{preview}

**Download:** {file_path}
"""
        return response_text
    except Exception as e:
        return f"Error: {e}"

In [9]:
def invoke_stream_model(question, model):

    payload = {
        "model": model,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question}
        ],
        "stream": True,
        "response_format": {"type": "json_object"}
    }

    try:
        stream = openrouter.chat.completions.create(**payload)

        collected_response = ""

        # Stream tokens
        for chunk in stream:

            if not chunk.choices:
                continue

            delta = chunk.choices[0].delta

            if hasattr(delta, "content") and delta.content:
                token = delta.content
                collected_response += token

                yield collected_response

        # Parse JSON safely
        try:
            schema_info = json.loads(collected_response)
        except json.JSONDecodeError:
            yield " Model returned invalid JSON."
            return

        # Validate schema
        if "schema" not in schema_info:
            yield f"Invalid schema response:\n\n{schema_info}"
            return

        rows = schema_info.get("rows", 100)
        schema = schema_info["schema"]

        # Generate dataset
        df, file_path = generate_dataset(schema, rows)

        preview = df.head(10).to_markdown()

        final_output = f"""
## Dataset Generated Successfully

**Total Rows:** {rows}

### Schema
```json
{json.dumps(schema, indent=2)}
```

{preview}

**Download:** {file_path}
"""
        yield final_output
    except Exception as e:
        yield f"Error: {e}"

In [18]:

def router(question, model, mode):
    if mode == "Streaming":
        # Yield from streaming generator
        yield from invoke_stream_model(question, model)
    else:
        # Wrap normal response in generator
        yield invoke_model(question, model)

In [ ]:
import gradio as gr

message_input = gr.Textbox(
    label="Your message",
    info="Describe the dataset you want",
    placeholder="Example: Generate a dataset of 1000 fintech transactions",
    lines=7,
    submit_btn="Send"
)

model_selector = gr.Dropdown(
    choices=models,
    label="Select model",
    value=models[0]
)

mode_selector = gr.Radio(
    ["Normal", "Streaming"],
    value="Streaming",
    label="Response Mode"
)

message_output = gr.Markdown(label="Response")


def router(message, history, model, mode):
    
    if mode == "Streaming":
        for chunk in invoke_stream_model(message, model):
            yield chunk

    else:
        response = invoke_model(message, model)
        return response


view = gr.ChatInterface(
    fn=router,
    title="Synthetic Dataset Generator AI",
    description="Describe the dataset you want and download the generated synthetic data.",
    
    additional_inputs=[model_selector, mode_selector],

    textbox=message_input,

    submit_btn="Send",

    examples=[
        ["Generate a dataset for an ecommerce store"],
        ["Generate a dataset for a healthcare provider"]
    ],

    flagging_mode="never"
)

view.launch(share=True)

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.
